In [123]:
import torch as torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4
eval_iters = 250


cpu


In [124]:
with open('wizard_of_oz.txt', 'r', encoding='utf=8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)
print(chars)
print(vocabulary_size)

['\n', ' ', '!', '#', '$', '%', '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
86


In [125]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [126]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [127]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('input is ', context, ' when target is ', target)

input is  tensor([29])  when target is  tensor(60)
input is  tensor([29, 60])  when target is  tensor(53)
input is  tensor([29, 60, 53])  when target is  tensor(68)
input is  tensor([29, 60, 53, 68])  when target is  tensor(72)
input is  tensor([29, 60, 53, 68, 72])  when target is  tensor(57)
input is  tensor([29, 60, 53, 68, 72, 57])  when target is  tensor(70)
input is  tensor([29, 60, 53, 68, 72, 57, 70])  when target is  tensor(1)
input is  tensor([29, 60, 53, 68, 72, 57, 70,  1])  when target is  tensor(35)


In [128]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([223953]) torch.int64
tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [129]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    dat = train_data if split == 'train' else val_data
    ix = torch.randint(len(dat) - block_size, (batch_size,))
    x = torch.stack([dat[i:i+block_size]     for i in ix])
    y = torch.stack([dat[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch('train')
print('inputs')
print(x)
print('targets')
print(y)



inputs
tensor([[55, 57,  1, 54, 57, 72, 75, 57],
        [60, 72, 58, 73, 64, 64, 77, 10],
        [ 1, 75, 61, 71, 60,  1, 72, 67],
        [67, 75,  1, 61, 72, 10, 83,  1]])
targets
tensor([[57,  1, 54, 57, 72, 75, 57, 57],
        [72, 58, 73, 64, 64, 77, 10,  1],
        [75, 61, 71, 60,  1, 72, 67,  1],
        [75,  1, 61, 72, 10, 83,  1, 70]])


In [130]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [152]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # ottengo la parte "indovinata"
            logits, loss = self.forward(index)
            # mi concentro solo sull'ultimo step
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocabulary_size)
m = model.to(device)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
            


™4UfHS”PIyRK2-ip-0:1O!FfD;cQ™”u%?XG ieIWJl;3;xALkU/•jdHe/hZIA•SE,14EUj.tVr’BT$-!jbwLsDYc664“MSDzek%D2PgkLI88I4y(h$—XXC—R4R‘gM96k•23cu*FKC.;SaJFf.E•:‘Vb(D!.93!o7v0a9DS/8pS8%og#rr/mi%js5sH84Fq6gk“$ ..L+M:oCm•“ODSD/y8sI+cCTar%ZIL?vY9-wVrFf6opIab02QPij*Z7sjqNRREX+)%)1hTru,KJ”Q9$#K2o™:cCae$f%970Vvd$)FKI8
4j#?P/G’g:bwx—(lzAlK.J*VPb+vrxoR%e2$Ug88i)BV4:atJL—’IJ!x R’z B
™”!-8“—q$C)F,W
™d$x——’%h
PRs•?Zr———
4KGwPx Q/23K.tn•’*LO0gKfh;Vh$H’ecwyvDa8pq!DNfgwPuK“I+vqX%BlmBT/•Ed)-“mZALtX6Jqw+Apt( 0gk+m%aIcTO/•7Z


In [161]:
#creo un optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}, train loss {losses['train']:.4f}, val loss : {losses['val']:.4f}")

    #prendo un apiccola parte di data
    xb, yb = get_batch('train')

    #valutazione loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


step 0, train loss 2.4280, val loss : 2.6823
step 250, train loss 2.4012, val loss : 2.6709
step 500, train loss 2.4202, val loss : 2.7337
step 750, train loss 2.4166, val loss : 2.7136
step 1000, train loss 2.4302, val loss : 2.7382
step 1250, train loss 2.4137, val loss : 2.6836
step 1500, train loss 2.3957, val loss : 2.6883
step 1750, train loss 2.4191, val loss : 2.6868
step 2000, train loss 2.4479, val loss : 2.6971
step 2250, train loss 2.3991, val loss : 2.6934
step 2500, train loss 2.4091, val loss : 2.7294
step 2750, train loss 2.3929, val loss : 2.6770
step 3000, train loss 2.4062, val loss : 2.6764
step 3250, train loss 2.3917, val loss : 2.6783
step 3500, train loss 2.4069, val loss : 2.7043
step 3750, train loss 2.4009, val loss : 2.6762
step 4000, train loss 2.3826, val loss : 2.7303
step 4250, train loss 2.3965, val loss : 2.7201
step 4500, train loss 2.3868, val loss : 2.6597
step 4750, train loss 2.3967, val loss : 2.7127
step 5000, train loss 2.4030, val loss : 2.710

In [162]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Lers, enouslo the youendondm futint atheg chtom w;rars ant

are the headoawas t asin swethe

sek
“I;d ginommieighede d wof where,”
“Aut s, bacrstoont we berethemay ScavF‘, f behindas there, Win hiced od gr ftathid so llly. artrt The an sher thaife s itor nd CEM?t sedue?” omand ad, w frere omashend hesaherorveekm;3ulecky, nd
thas mshy ow, cly of t?.

berap Thould aisus h ig;yil,” ifthar a bbe heds they ariny, aneres, OXCgo
hed tty.” te by w ta, apouapun my;Wousoflothysig bed heaklld wawarerlllepo
